# RL

# Setup

In [ ]:
from thesis_project import (
    config,
)

config.default()

TICKER: config.FutTicker = "fbtp"

# Lobs directories
LOB_DIR = config.DATA_RAW_DIR / TICKER
LOB_FUT_DIR = LOB_DIR / "fut"
LOB_CTD_DIR = LOB_DIR / "ctd"

# NOTE: Use "notebook-name" if needed
NOTEBOOK = None

# Output directories
if NOTEBOOK is not None:
    OUT_FIG_DIR = config.RES_FIG_DIR / TICKER / NOTEBOOK
    OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
    OUT_TAB_DIR = config.RES_TAB_DIR / TICKER / NOTEBOOK
    OUT_TAB_DIR.mkdir(parents=True, exist_ok=True)
    OUT_EXP_DIR = config.RES_EXP_DIR / TICKER / NOTEBOOK
    OUT_EXP_DIR.mkdir(parents=True, exist_ok=True)

# Common

In [ ]:
import json

from thesis_project.rl import report

# Main

## Baselines

In [ ]:
from thesis_project import rl
from thesis_project.dataset.features_market import FeatureSpec
from thesis_project.dataset.features_transforms import Identity
from thesis_project.rl.env_factory import make_gym_env

market: list[FeatureSpec] = [
    FeatureSpec(base_id="basis", transforms=[Identity()]),
]

dataset_config = rl.dataset.DatasetConfig(
    ticker="fbtp",
    n_jobs_market=1,
    contract_mode="round",
    market_set="dummy",
    calendar_set="dummy",
    calendar_enc_set="dummy",
)

dummy_dataset = rl.dataset.build_rl_dataset(dataset_config)

In [ ]:
env_config_serial = rl.env.EnvConfig(
    reset_mode="serial",
    price_mode="mid",
    persistence_min=10,
    trajectory_min=0,
)

### Long Only

In [ ]:
env = make_gym_env(
    dummy_dataset,
    env_config_serial,
)

fixed_long_records = rl.evaluation.evaluate_model_gym(
    predictor=rl.policies.FixedLongPolicy(),
    env=env,
)

In [ ]:
metadata = {"which": "both"}
report.report({0: fixed_long_records}, metadata=metadata)

### Short Only

In [ ]:
env = make_gym_env(dummy_dataset, env_config_serial)

fixed_short_records = rl.evaluation.evaluate_model_gym(
    predictor=rl.policies.FixedShortPolicy(),
    env=env,
)

In [ ]:
metadata = {"which": "both"}
report.report({0: fixed_short_records}, metadata=metadata)

### Random

In [ ]:
import pandas as pd

from thesis_project.utils.misc import generate_seeds

seed = 42
seeds = generate_seeds(n=5, seed=seed)

random_records: dict[int, pd.DataFrame] = {}
for seed in seeds:
    env = make_gym_env(dataset=dummy_dataset, env_config=env_config_serial)
    policy = rl.policies.RandomPolicy(seed=seed)
    random_records[seed] = rl.evaluation.evaluate_model_gym(predictor=policy, env=env)

In [ ]:
metadata = {"which": "both"}
report.report(random_records, metadata=metadata)

## Single Experiment

In [ ]:
# TODO:
# - Try 2x, 5x, 10x CTD
# - Try different persistence: 1min, 30min, 60min


run = 6
batch_size = 256
clip_range = 0.2

root = config.RES_EXP_DIR / TICKER / "rl"
folder = f"run-{run:03d}"
path = root / folder

with open(path / "config.json", "r") as f:
    settings = json.load(f)

records_train = report.load_records(
    path,
    batch_size=batch_size,
    clip_range=clip_range,
    seeds=settings["seeds"],
    split="train",
)

records_test = report.load_records(
    path,
    batch_size=batch_size,
    clip_range=clip_range,
    seeds=settings["seeds"],
    split="test",
)

### Training set

In [ ]:
report.report(records_train)

### Test Set

In [ ]:
report.report(records_test)

### Spread considerations

In [ ]:
# from thesis_project.dataset.data import BASE_FEATURES
# from thesis_project.utils.io import load_filtered_parquet

# pnl_mean = 76_439.38
# pnl_std = 17_757.55
# n_trades_mean = 786.20
# n_trades_std = 366.64

# max_spread = pnl_mean / n_trades_mean

# spread_basis = records[seeds[0]].set_index("timestamp")["basis_spread"]

# spread_ctd = load_filtered_parquet(BASE_FEATURES["ctd_spread"].path).loc[spread_basis.index]
# spread_ctd *= 2_000_000 / 100
# spread_fut = load_filtered_parquet(BASE_FEATURES["fut_spread"].path).loc[spread_basis.index]
# spread_fut *= 100_000 / 100

In [ ]:
# n = 22
# print(f"{'Max spread per trade:':<{n}} {max_spread:,.2f} €")
# print(f"{'Basis spread:':<{n}} {spread_basis.mean() / 2:,.2f} ± {spread_basis.std() / 2:,.2f} €")
# print(f"{'FBTP CTD spread:':<{n}} {spread_ctd.mean() / 2:,.2f} ± {spread_ctd.std() / 2:,.2f} €")
# print(
#     f"{'FBTP FUT spread:':<{n}} {15.5 * spread_fut.mean() / 2:,.2f} ± {18 * spread_fut.std() / 2:,.2f} €"
# )

## Compare Experiments

In [ ]:
run_a = 5
run_b = 6

batch_size = 512
clip_range = 0.3

root = config.RES_EXP_DIR / TICKER / "rl"

# Report A
path_a = root / f"run-{run_a:03d}"
with open(path_a / "config.json") as f:
    settings_a = json.load(f)

records_a = report.load_records(
    path_a,
    batch_size=batch_size,
    clip_range=clip_range,
    seeds=settings_a["seeds"],
    split="test",
)

metadata_a = report.load_metadata(
    path_a,
    batch_size=batch_size,
    clip_range=clip_range,
    seeds=settings_a["seeds"],
    split="test",
)

# Report B
path_b = root / f"run-{run_b:03d}"
with open(path_b / "config.json") as f:
    settings_b = json.load(f)

records_b = report.load_records(
    path_b,
    batch_size=batch_size,
    clip_range=clip_range,
    seeds=settings_b["seeds"],
    split="test",
)

metadata_b = report.load_metadata(
    path_b,
    batch_size=batch_size,
    clip_range=clip_range,
    seeds=settings_b["seeds"],
    split="test",
)

In [ ]:
report.compare_report(
    records_a,
    records_b,
    metadata_a,
    metadata_b,
    label_a=f"Run {run_a}",
    label_b=f"Run {run_b}",
    which="net",
)